# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', None)}\n")
print(f"Description:\n{getattr(metadata, 'description', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's explore which record sets are available in the dataset. We enumerate by their unique `@id` identifiers, following Croissant schema conventions.

In [ ]:
# List all record sets by @id, name, and fields
record_sets = dataset.record_sets

for rs in record_sets:
    print(f"RecordSet @id: {getattr(rs, '@id', None)}")
    print(f"  Name: {getattr(rs, 'name', None)}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print(f"  Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"    Field @id: {getattr(field, '@id', None)} | Name: {getattr(field, 'name', None)} | DataType: {getattr(field, 'data_type', None)}")
    print('-' * 60)
# List all record set IDs for later convenience
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We use the exact `@id` values of record sets and fields from above. Here we extract all available record sets.

In [ ]:
# Extract all record sets to DataFrames, using @id for access
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set @id: {record_set_id} ({df.shape[0]} rows)")

# Preview the fields of the first loaded record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nFields in DataFrame for record set @id: {first_rs_id}")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. We demonstrate filtering, normalization, and grouping using a numeric field and a group field from one of the record sets.

Replace selections below with the desired field `@id`s from your overview.

In [ ]:
# Select record set and field @id's for EDA

# Use the first record set by default (adjust as needed)
record_set_id = first_rs_id  # e.g., 'https://api.app.sen.science/frontiers/7862866/rs-cancer_cases'
df = dataframes[record_set_id]

# Find candidate numeric and group fields by inspecting columns
print("DataFrame columns:", df.columns.tolist())

# You may need to inspect the columns to choose appropriate field IDs; for example:
# numeric_field_id = '@id-of-numeric-field'  # e.g., 'age_at_second_diagnosis'
# group_field_id = '@id-of-group-field'      # e.g., 'sex'

# For demonstration, use the first numeric column that is not an identifier:
import numpy as np

numeric_field_id = None
for c in df.columns:
    # Guess numeric field (actual field name, not @id, adjust as needed)
    if df[c].dtype in (np.int64, np.float64) and 'id' not in c.lower():
        numeric_field_id = c
        break

if numeric_field_id is None:
    print("No obvious numeric field found for EDA.")
else:
    print(f"Using numeric field: {numeric_field_id}")

    # Set a threshold for filtering (arbitrary for demonstration)
    threshold = df[numeric_field_id].mean()  # use mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()

    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by another categorical field
    # Guess a suitable group field
    group_field = None
    for c in df.columns:
        if df[c].dtype == object and c != numeric_field_id and 'id' not in c.lower():
            group_field = c
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize distributions and relationships using matplotlib or seaborn. Replace field selections as appropriate for your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} By {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('No numeric field identified for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to connect to a FAIR-compliant dataset described by a Croissant schema using the `mlcroissant` library, explored its structure and metadata using universally unique `@id` references, extracted and processed records with Python pandas, performed initial exploratory analysis and visualizations. This process can be extended and customized for further modeling or domain-specific inference.

Remember to double-check field IDs against the actual record set overview above for best field choices in your analysis.

_For further exploration visit [mlcroissant GitHub](https://github.com/mlcommons/croissant)._